**Hashim Zia (hz2776) Final Notebook for midterm.**

I trained Qwen2.5-1.5B-Instruct on this notebook which ended up being my best model. This notebook only contains the process of acheiving the best results I obtained. My report contains details of all the different models and ablations I tried. This is adapted from the unsloth starter notebook.

In [ ]:
#This import imports older transformers which works with Qwen2.5, Qwen3.5 only works with newer transformers package but that breaks 2.5.
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install  transformers
%pip install --no-deps trl==0.22.2

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from unsloth import FastLanguageModel
import numpy as np
import pandas as pd
import torch
from datasets import concatenate_datasets, load_dataset, Dataset
from tqdm.auto import tqdm
from datasets import Dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:

CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "max_seq_length": 560,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 100,
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2.5",
}

model, tokenizer = FastLanguageModel.from_pretrained(
    "/kaggle/input/models/hashimzia1/qwen2-5best/pytorch/default/2/qwen3b-ci"
)

In [ ]:
#UNCOMMENT THIS BLOCK FOR FINETUNING

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=CONFIG["model_name"],
#     max_seq_length=CONFIG["max_seq_length"],
#     dtype=None,
#     load_in_4bit=True,
# )

# model = FastLanguageModel.get_peft_model(
#     model,
#     r=CONFIG["lora_r"],
#     lora_alpha=CONFIG["lora_alpha"],
#     lora_dropout=0,
#     bias="none",
#     target_modules=[
#         "q_proj", "k_proj", "v_proj", "o_proj",
#         "gate_proj", "up_proj", "down_proj",
#     ],
#     use_gradient_checkpointing="unsloth",
#     # random_state=SEED,
# )

The below code takes the input as the 50,000 training samples and filters out only inputs that are within 512 tokens. I explain the reasoning behind this in my report.

In [ ]:
TRAIN_CSV_PATH = "/kaggle/input/datasets/hashimzia1/d12345/train.csv"
OUTPUT_CSV_PATH = "/kaggle/working/train_filter_512.csv"
PROMPT_COL = "prompt"
SVG_COL = "svg"
MAX_SEQ_LENGTH = 512
ET.register_namespace("", "http://www.w3.org/2000/svg")
def format_svg_sample(prompt: str, svg_code: str) -> str:
    """Format a prompt-SVG pair into the chat template expected by the model."""
    messages = [
        {"role": "system", "content": "You are an expert SVG code generator. Generate clean, valid SVG code based on the user's description."},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": svg_code},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

def extract_svg_block(text: str) -> str:
    if not isinstance(text, str):
        return ""
    m = re.search(r"(<svg[\s\S]*?</svg>)", text, re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()

def token_length_for_example(prompt: str, svg_text: str) -> int:
    formatted = format_svg_sample(prompt, svg_text)
    token_ids = tokenizer(formatted, add_special_tokens=False)["input_ids"]
    return len(token_ids)

df = pd.read_csv(TRAIN_CSV_PATH)
print("Original rows:", len(df))
print("Columns:", list(df.columns))

df = df[[PROMPT_COL, SVG_COL]].dropna().reset_index(drop=True)

kept_rows = []
dropped_reasons = {
    "too_long_for_512": 0,
}

for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt = row[PROMPT_COL]
    raw_svg = row[SVG_COL]

    svg = extract_svg_block(raw_svg)
    
    n_tokens = token_length_for_example(prompt, svg)
    
    if n_tokens > MAX_SEQ_LENGTH:
        dropped_reasons["too_long_for_512"] += 1
        continue

    kept_rows.append({
        PROMPT_COL: prompt,
        SVG_COL: svg,
        "token_length": n_tokens,
    })

filtered_df = pd.DataFrame(kept_rows)

print("\n=== RESULT ===")
print("Kept rows:", len(filtered_df))
print("Dropped rows:", len(df) - len(filtered_df))
print("Dropped reasons:", dropped_reasons)

if len(filtered_df):
    print("\nToken length stats on kept rows:")
    print(filtered_df["token_length"].describe())

print(filtered_df)
filtered_df[[PROMPT_COL, SVG_COL]].to_csv(OUTPUT_CSV_PATH, index=False)
print("\nSaved filtered CSV to:", OUTPUT_CSV_PATH)

def build_text(example):
    return {
        "text": format_svg_sample(example[PROMPT_COL], example[SVG_COL])
    }

train_df_transformed = filtered_df[[PROMPT_COL, SVG_COL]].copy()

train_dataset_transformed = Dataset.from_pandas(
    train_df_transformed,
    preserve_index=False
).map(build_text, remove_columns=[PROMPT_COL, SVG_COL])

print("\nTransformed HF dataset size:", len(train_dataset_transformed))
print("\nSample text preview:\n")
print(train_dataset_transformed[0]["text"][:2500])

In [ ]:
train_raw = pd.read_csv("/kaggle/working/train_filter_512.csv")
train_raw = train_raw[["prompt", "svg"]].dropna()
train_raw = train_raw.drop_duplicates(subset=["prompt", "svg"]).reset_index(drop=True)

train_part,val_part = train_test_split(
    train_raw,
    test_size=0.02,
    random_state=42,
    shuffle=True,
)
train_ds = Dataset.from_pandas(train_part[["prompt", "svg"]], preserve_index=False)
eval_ds   = Dataset.from_pandas(val_part[["prompt", "svg"]], preserve_index=False)

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[3]

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}

train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

In [ ]:
#UNCOMMENT THIS BLOCK FOR FINE TUNING

# from transformers import TrainingArguments
# from trl import SFTTrainer
# training_args = TrainingArguments(
#     output_dir=CONFIG["output_dir"],
#     num_train_epochs=CONFIG["num_train_epochs"],
#     per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
#     gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
#     learning_rate=CONFIG["learning_rate"],
#     warmup_ratio=CONFIG["warmup_ratio"],
#     weight_decay=CONFIG["weight_decay"],
#     fp16=not torch.cuda.is_bf16_supported(),
#     bf16=torch.cuda.is_bf16_supported(),
#     logging_steps=CONFIG["logging_steps"],
#     eval_strategy="steps",
#     eval_steps=CONFIG["eval_steps"],
#     save_steps=CONFIG["save_steps"],
#     save_total_limit=2,
#     report_to="none",
#     optim="paged_adamw_8bit",
#     lr_scheduler_type="cosine",
#     seed=SEED,
#     average_tokens_across_devices=False
# )
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_text,
#     eval_dataset=eval_text,
#     dataset_text_field="text",
#     max_seq_length=CONFIG["max_seq_length"],
#     packing=True,
#     args=training_args,
# )
# train_result = trainer.train()
# train_result

In [ ]:
# os.makedirs(CONFIG["output_dir"], exist_ok=True)
# trainer.save_model(CONFIG["output_dir"])
# tokenizer.save_pretrained(CONFIG["output_dir"])

# print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
FastLanguageModel.for_inference(model)

TEST_CSV_PATH = "/kaggle/input/datasets/hashimzia1/d12345/test.csv"
OUTPUT_CSV_PATH = "/kaggle/working/submission_raw.csv"
tokenizer.padding_side = "left"
ID_COL = "id"
PROMPT_COL = "prompt"
BATCH_SIZE = 8
GENERATE_TOKENS = 580
SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Generate clean, valid SVG code based on the user's description."
)

model.eval()

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = True

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

def extract_svg_block(text: str) -> str:
    if not isinstance(text, str):
        return ""
    m = re.search(r"(<svg[\s\S]*?</svg>)", text, re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()

def is_complete_svg(text: str) -> bool:
    return isinstance(text, str) and bool(re.search(r"<svg[\s\S]*?</svg>", text, re.IGNORECASE))

def build_chat_text(prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def generate_svg_batch(prompts, max_new_tokens=512):
    texts = [build_chat_text(p) for p in prompts]

    inputs = tokenizer(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    input_lengths = inputs["attention_mask"].sum(dim=1).tolist()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05,
            use_cache=True,
        )

    results = []
    for i in range(outputs.shape[0]):
        gen_tokens = outputs[i][int(input_lengths[i]):]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        results.append(extract_svg_block(decoded))

    return results

test_df = pd.read_csv(TEST_CSV_PATH)
print("Test shape:", test_df.shape)
print(test_df.head())

submission_rows = []
incomplete_first_pass = 0
retried_second_pass = 0

for start in tqdm(range(0, len(test_df), BATCH_SIZE)):
    batch_df = test_df.iloc[start:start + BATCH_SIZE]
    prompts = batch_df[PROMPT_COL].tolist()
    ids = batch_df[ID_COL].tolist()

    svgs = generate_svg_batch(prompts, max_new_tokens=GENERATE_TOKENS)
    for sample_id, svg in zip(ids, svgs):
        submission_rows.append({
            "id": sample_id,
            "svg": svg,
        })
submission_df = pd.DataFrame(submission_rows)[["id", "svg"]]
submission_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved submission to: {OUTPUT_CSV_PATH}")
print(submission_df.head())

# for i in range(min(3, len(submission_df))):
#     print("=" * 100)
#     print("ID:", submission_df.iloc[i]["id"])
#     print("PROMPT:", test_df.iloc[i][PROMPT_COL])
#     print("SVG:")
#     print(submission_df.iloc[i]["svg"][:1500])
#     print()

In [ ]:
SVG_CSV_PATH = "/kaggle/working/submission_raw.csv"
PROMPT_CSV_PATH = "/kaggle/input/datasets/hashimzia1/d12345/test.csv"
INVALID_REPORT_PATH = "/kaggle/working/invalid_svgs.csv"
MATCHED_OUTPUT_PATH = "/kaggle/working/submission.csv"

ID_COL = "id"
SVG_COL = "svg"
PROMPT_COL = "prompt"

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter"
}

DISALLOWED_TAGS = {
    "script", "foreignObject", "animate", "animateMotion",
    "animateTransform", "set"
}


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def local_tag(tag: str) -> str:
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


def extract_svg_block(text: str) -> str:
    if not isinstance(text, str):
        return ""
    m = re.search(r"(<svg[\s\S]*?</svg>)", text, re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


def is_valid_svg(svg_text: str):
    if not isinstance(svg_text, str):
        return False, "not_string"

    svg_text = extract_svg_block(svg_text)

    if not svg_text:
        return False, "empty"

    if len(svg_text) > MAX_SVG_LENGTH:
        return False, "too_long"

    lowered = svg_text.lower()

    if re.search(r"\son[a-z]+\s*=", lowered):
        return False, "event_handler"

    if re.search(r'(?:xlink:href|href)\s*=\s*["\']https?://', lowered):
        return False, "external_ref"

    if "javascript:" in lowered:
        return False, "javascript_ref"

    try:
        root = ET.fromstring(svg_text)
    except Exception:
        return False, "invalid_xml"

    if local_tag(root.tag) != "svg":
        return False, "root_not_svg"

    tags = [local_tag(el.tag) for el in root.iter()]

    if any(tag in DISALLOWED_TAGS for tag in tags):
        return False, "disallowed_tag"

    if any(tag not in ALLOWED_TAGS and tag not in DISALLOWED_TAGS for tag in tags):
        return False, "unknown_tag"

    path_count = sum(1 for el in root.iter() if local_tag(el.tag) == "path")
    if path_count > MAX_PATH_COUNT:
        return False, "too_many_paths"

    return True, "ok"


svg_df = pd.read_csv(SVG_CSV_PATH)
print("Loaded SVG CSV:", svg_df.shape)

valid_svg_map = {}
invalid_rows = []
replacement_reasons = {}

for idx, row in svg_df.iterrows():
    item_id = str(row[ID_COL]).strip() if pd.notna(row[ID_COL]) else ""
    svg = row[SVG_COL]

    ok, reason = is_valid_svg(svg)

    if ok and item_id:
        valid_svg_map[item_id] = extract_svg_block(svg)
    else:
        invalid_rows.append({
            "row_number": idx + 2,  # +2 because header is row 1
            "id": item_id,
            "reason": reason if not ok else "missing_id"
        })
        replacement_reasons[reason if not ok else "missing_id"] = (
            replacement_reasons.get(reason if not ok else "missing_id", 0) + 1
        )

invalid_df = pd.DataFrame(invalid_rows)
invalid_df.to_csv(INVALID_REPORT_PATH, index=False)

print(f"Valid SVGs kept: {len(valid_svg_map)}")
print(f"Invalid SVG rows: {len(invalid_rows)}")
print(f"Invalid report saved to: {INVALID_REPORT_PATH}")

if replacement_reasons:
    print("Invalid reasons:")
    for k, v in sorted(replacement_reasons.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v}")


prompt_df = pd.read_csv(PROMPT_CSV_PATH)
print("Loaded prompt CSV:", prompt_df.shape)

assert ID_COL in prompt_df.columns, f"Missing column: {ID_COL}"
assert PROMPT_COL in prompt_df.columns, f"Missing column: {PROMPT_COL}"

output_rows = []

matched = 0
fallback_used = 0

for _, row in prompt_df.iterrows():
    item_id = str(row[ID_COL]).strip() if pd.notna(row[ID_COL]) else ""
    prompt = row[PROMPT_COL] if pd.notna(row[PROMPT_COL]) else ""

    svg = valid_svg_map.get(item_id)
    if svg is not None:
        matched += 1
    else:
        svg = fallback_svg(prompt)
        fallback_used += 1

    output_rows.append({
        ID_COL: item_id,
        SVG_COL: svg
    })

output_df = pd.DataFrame(output_rows)
output_df[[ID_COL, SVG_COL]].to_csv(MATCHED_OUTPUT_PATH, index=False)

print(f"Matched rows: {matched}")
print(f"Fallback rows: {fallback_used}")
print(f"Saved matched output to: {MATCHED_OUTPUT_PATH}")
print(output_df.head())